# kona-ebm quickstart

Generates a small Sudoku dataset, builds an untrained "demo" encoder/energy/decoder stack, solves a single instance via latent-optimization inference, and plots the energy curve.

This notebook is meant to be run after `pip install -e ".[dev]"` from the repo root.

In [1]:
from kona_ebm.datasets import get_domain
from kona_ebm.inference import optimize_candidate, plot_energy_curve, solve
from kona_ebm.models import build_demo_bundle

domain = get_domain("sudoku", n_remove=45)
instance = domain.generate(1, seed=0)[0]
print("Puzzle (0 = blank):")
for row in instance.problem:
    print(row)

Puzzle (0 = blank):
[7, 9, 1, 0, 0, 0, 5, 0, 0]
[3, 5, 0, 0, 0, 6, 0, 0, 0]
[2, 6, 8, 0, 5, 1, 0, 3, 4]
[4, 3, 0, 0, 6, 5, 0, 0, 0]
[8, 0, 6, 0, 7, 0, 3, 4, 0]
[9, 0, 0, 0, 1, 3, 0, 0, 0]
[0, 0, 0, 0, 0, 7, 2, 0, 0]
[0, 0, 0, 5, 4, 2, 0, 7, 9]
[5, 7, 0, 0, 0, 0, 0, 0, 3]


In [2]:
encoder, energy, decoder = build_demo_bundle(domain, seed=0)

result = solve(domain, instance.problem, encoder, energy, decoder=decoder, max_iters=200, n_starts=4, max_restarts=3)
print("valid:", result.valid, "violations:", result.violations, "final_energy:", result.final_energy)
print("iterations:", result.n_iters, "restarts:", result.n_restarts, "runtime (s):", round(result.runtime_s, 3))
print()
print("Solution:")
for row in result.solution:
    print(row)

valid: False violations: 6 final_energy: 7.116644382476807
iterations: 2400 restarts: 3 runtime (s): 8.483

Solution:
[7, 9, 1, 3, 8, 4, 5, 2, 6]
[3, 5, 4, 7, 2, 6, 9, 9, 1]
[2, 6, 8, 9, 5, 1, 7, 3, 4]
[4, 3, 7, 8, 6, 5, 9, 9, 2]
[8, 1, 6, 2, 7, 9, 3, 4, 5]
[9, 2, 5, 4, 1, 3, 8, 6, 7]
[6, 4, 9, 1, 3, 7, 2, 5, 8]
[1, 8, 3, 5, 4, 2, 6, 7, 9]
[5, 7, 2, 6, 9, 8, 4, 1, 3]


## Visualizing the optimization trajectory

Re-run a single optimization (not the full multi-start solve) with trajectory recording on, then plot energy vs. iteration.

In [3]:
import torch

problem_t = domain.encode_problem(instance.problem).unsqueeze(0)
with torch.no_grad():
    latent = encoder(problem_t)
init = domain.random_solution(instance.problem).unsqueeze(0)

opt_result = optimize_candidate(energy, problem_t, latent, init, method="adam", lr=0.3, max_iters=150, record_trajectory=True)
plot_energy_curve(opt_result.energy_history, "energy_curve.png")
print("Saved energy_curve.png -- final energy:", opt_result.energy_history[-1])

Saved energy_curve.png -- final energy: 0.2823193669319153


## Next steps

- Train a real model: `python scripts/train.py dataset=sudoku model=mlp loss=contrastive training.max_epochs=20`
- Compare against baselines: `python scripts/run_benchmarks.py --domain sudoku --n_test 20`
- See `docs/inference_guide.md` and `docs/training_guide.md` for details.